# DBO_Quant — Portfolio Optimization

Run this notebook on a remote or on-prem computer. CPU is the default solver; GPU is optional.

Workflow: **Databricks inputs → Mean-CVaR optimization/frontier → backtest → optional rebalancing → persist results to DBO_Quant**.


In [ ]:
from pathlib import Path
import os, sys

DBO_QUANT_ROOT = Path(os.environ.get('DBO_QUANT_ROOT', Path.cwd())).resolve()
for candidate in [DBO_QUANT_ROOT, *DBO_QUANT_ROOT.parents]:
    if (candidate / 'optimization' / 'portfolio_optimization').exists() and (candidate / 'nvidia_bridge').exists():
        DBO_QUANT_ROOT = candidate
        break
sys.path.insert(0, str(DBO_QUANT_ROOT))

from optimization.portfolio_optimization.config import load_external_config
from optimization.portfolio_optimization.runner import run_external_workflow
print('DBO_Quant root:', DBO_QUANT_ROOT)


## Configuration
Edit `optimization/portfolio_optimization/portfolio_config.toml`. Set `solver = "cpu"` for CVXPY + CLARABEL or `solver = "gpu"` for CVXPY + NVIDIA cuOpt. CPU is the committed default. A local `.env` is optional for Databricks connection hints only.


In [ ]:
CONFIG = load_external_config(DBO_QUANT_ROOT)
for key, value in CONFIG.items():
    if key != 'profile':
        print(f'{key}: {value}')


In [ ]:
result = run_external_workflow(**CONFIG)

display(result['optimal_weights'].sort_values(ascending=False).to_frame())
display(result['frontier'].head(CONFIG['frontier_points']))
display(result['frontier_figure'])
display(result['backtest_results'])
if result['rebalance_results'] is not None:
    display(result['rebalance_results'])

print('optimization_run_id =', result['optimization_run_id'])
print('rebalance_run_id =', result['rebalance_run_id'])


## Next
Back in Databricks, open `notebooks/portfolio/03_OPTIMIZATION_RESULTS.py`, or use `notebooks/portfolio/02_MONTE_CARLO.py` with `source_type=optimization_run`. Persisted charts are also available through the OpenBB backend.
